# 13. Monte Carlo Szimuláció és Kockázatelemzés

**Cél:** a lakásbefektetések jövőbeli kockázatainak (hozamingadozás, üresedés, kamatkockázat) kvantifikálása sztochasztikus modellezéssel.

---

### 📖 Miért nem elég a determinisztikus számítás?
A statikus számítások feltételezik a folyamatos bérbeadottságot és a fix kamatot; a valós gazdasági környezet sztochasztikus. A Monte Carlo szimuláció több ezer, valószínűségi eloszlásokkal terhelt forgatókönyvet generál az extrém kockázatok feltérképezésére.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
elado = df[df['listing_type'] == 'elado']
kiado = df[df['listing_type'] == 'kiado']

base_price = elado['nm_ar_huf'].median() * 50  # 50 m² lakás vételár
base_rent = kiado['price_huf'].median() if not kiado.empty else 250000
np.random.seed(42)
print("Monte Carlo szimulációs motor kész.")

### 1. Kockázati Metrikák: VaR és CVaR

**📌 A Value at Risk értelmezése:**
- **VaR 95%:** 95% a valószínűsége, hogy az éves hozam ennél nem lesz rosszabb.
- **VaR 99%:** az extrém stressz-forgatókönyv határa.

**🔍 Gyakorlati konklúzió:**
A magas bruttó hozamú szegmensek volatilitása (üresedési kockázata) aszimmetrikusan magasabb lehet; a kockázattal súlyozott nettó jelenérték (NPV) gyakran a stabil, diverzifikált portfóliók felé billenti a mérleget.

In [ ]:
N_ITERS = 10000
import statsmodels.api as sm

# --- KALIBRÁCIÓ A TÉNYLEGES ADATOKBÓL (nem hardkódolt feltételezések) ---
# 1. Árvolatilitás: a hedonikus reziduumok szórása (a keresztmetszeti diszperzió
#    a piaci hozamvolatilitás konzervatív alsó becslése)
elado_c = elado.dropna(subset=['log_nm_ar', 'korrigalt_alapterulet_nm', 'szobaszam_osszes',
                               'allapot_kod', 'is_panel', 'has_lift']).copy()
if len(elado_c) > 30:
    _m_el = sm.OLS(elado_c['log_nm_ar'],
                   sm.add_constant(elado_c[['korrigalt_alapterulet_nm', 'szobaszam_osszes',
                                            'allapot_kod', 'is_panel', 'has_lift']])).fit()
    resid_std_price = float(_m_el.resid.std())
else:
    resid_std_price = 0.12
kiado_c = kiado.dropna(subset=['log_ar', 'korrigalt_alapterulet_nm']).copy()
if len(kiado_c) > 30:
    _m_ki = sm.OLS(kiado_c['log_ar'],
                   sm.add_constant(kiado_c[['korrigalt_alapterulet_nm']])).fit()
    resid_std_rent = float(_m_ki.resid.std())
else:
    resid_std_rent = 0.10

std_price_pct = float(np.clip(resid_std_price, 0.08, 0.20))
std_rent_pct = float(np.clip(resid_std_rent, 0.08, 0.20))
print(f"Kalibrált árvolatilitás: {std_price_pct*100:.1f}% (reziduális szórás: {resid_std_price:.3f})")
print(f"Kalibrált bérleti volatilitás: {std_rent_pct*100:.1f}% (reziduális szórás: {resid_std_rent:.3f})")

# 2. Ár-bérlet korreláció: területi cellákon (városrész × immissziós sáv) becsülve
def _cells_med(sub, col):
    g = sub.dropna(subset=[col]).groupby(['varosresz', 'vasut_zona'], observed=True)[col].median()
    return g
try:
    p_med = _cells_med(elado, 'nm_ar_huf')
    r_med = _cells_med(kiado, 'price_huf')
    both = pd.concat([p_med.rename('p'), r_med.rename('r')], axis=1).dropna()
    if len(both) >= 5:
        corr = float(np.clip(both['p'].corr(both['r']), 0.0, 0.95))
    else:
        corr = 0.65
except Exception:
    corr = 0.65
print(f"Kalibrált ár-bérlet korreláció (területi cellákon, N={0 if 'both' not in dir() else len(both)}): {corr:.3f}")

# Korrelált sztochasztikus sokkok
mean_vec = [base_price, base_rent]
std_price = base_price * std_price_pct
std_rent = base_rent * std_rent_pct
cov_matrix = [
    [std_price**2, corr * std_price * std_rent],
    [corr * std_price * std_rent, std_rent**2]
]
rng = np.random.default_rng(42)
corr_shocks = rng.multivariate_normal(mean_vec, cov_matrix, size=N_ITERS)
price_shocks = corr_shocks[:, 0]
rent_shocks = corr_shocks[:, 1]
occ_shocks = np.clip(rng.normal(loc=0.92, scale=0.06, size=N_ITERS), 0.70, 1.00)
disc_rate = 0.05
op_cost_ratio = 0.15

# 20 éves NPV számítás — a terminálérték KÜLÖN felértékelődési eloszlásból
eves_netto_cf = rent_shocks * 12 * occ_shocks * (1.0 - op_cost_ratio)
annuity_factor = (1.0 - (1.0 + disc_rate) ** -20) / disc_rate

appr_base = np.clip(rng.normal(loc=1.20, scale=0.10, size=N_ITERS), 0.95, 1.6)
appr_ren = np.clip(rng.normal(loc=1.80, scale=0.15, size=N_ITERS), 1.2, 2.4)
term_base = price_shocks * appr_base / ((1.0 + disc_rate) ** 20)
npv_base = (eves_netto_cf * annuity_factor + term_base) - price_shocks
term_ren = price_shocks * appr_ren / ((1.0 + disc_rate) ** 20)
npv_renewal = (eves_netto_cf * annuity_factor + term_ren) - price_shocks

# Stagflációs stressz
stress_occ = np.clip(rng.normal(loc=0.80, scale=0.08, size=N_ITERS), 0.50, 0.90)
stress_cost = 0.20
stress_disc = 0.07
stress_annuity = (1.0 - (1.0 + stress_disc) ** -20) / stress_disc
stress_cf = rent_shocks * 12 * stress_occ * (1.0 - stress_cost)
term_stress = price_shocks * 0.90 / ((1.0 + stress_disc) ** 20)
npv_stress = (stress_cf * stress_annuity + term_stress) - price_shocks

mean_base = np.mean(npv_base)
se_base = np.std(npv_base) / np.sqrt(N_ITERS)
prob_base = (npv_base > 0).mean() * 100
mean_ren = np.mean(npv_renewal)
se_ren = np.std(npv_renewal) / np.sqrt(N_ITERS)
prob_ren = (npv_renewal > 0).mean() * 100
mean_stress = np.mean(npv_stress)
prob_stress = (npv_stress > 0).mean() * 100
print(f"MC-hiba (SE az átlagra): konzervatív ±{se_base/1e6:.2f} M Ft, megújítás ±{se_ren/1e6:.2f} M Ft.")

kpi_cards = [
    ("Alap Vételár (50 m²)", fmt_mft(base_price / 1e6), "Referencia lakás", "#7c3aed"),
    ("Konzervatív NPV", fmt_mft(mean_base / 1e6), f"P(NPV>0): {prob_base:.1f}% | SE: ±{se_base/1e6:.2f} M Ft", "#ef4444"),
    ("Városmegújítás NPV", fmt_mft(mean_ren / 1e6), f"P(NPV>0): {prob_ren:.1f}% | SE: ±{se_ren/1e6:.2f} M Ft", "#10b981"),
    ("Stagflációs NPV", fmt_mft(mean_stress / 1e6), f"P(NPV>0): {prob_stress:.1f}%", "#dc2626"),
    ("VaR 95% (Konzervatív)", fmt_mft(np.percentile(npv_base, 5) / 1e6), "5%-os kvantilis", "#d97706"),
    ("Kalibrált árvolatilitás", f"{std_price_pct*100:.1f}%", "Hedonikus reziduumokból", "#059669")
]
display(HTML(kpi_grid_html(kpi_cards)))


### 2. Kumulatív Eloszlásfüggvény (CDF) és Konfidencia Sáv

**📌 Hogyan olvassuk a kumulatív görbét?**
Az S-görbe megmutatja annak a valószínűségét, hogy a hozam egy adott érték alatt marad. A meredek felfutás alacsony kimeneteli szórást és magas stabilitást igazol.

In [ ]:
fig1 = go.Figure()
fig1.add_trace(go.Histogram(
    x=npv_base / 1e6, nbinsx=50, name=f'Konzervatív bérlet (P>0: {prob_base:.1f}%)',
    marker_color='#ef4444', opacity=0.65
))
fig1.add_trace(go.Histogram(
    x=npv_renewal / 1e6, nbinsx=50, name=f'Városmegújítás Total Return (P>0: {prob_ren:.1f}%)',
    marker_color='#10b981', opacity=0.65
))
fig1.add_vline(x=0, line_color='black', line_width=2, line_dash='dash', annotation_text='NPV = 0')
fig1.update_layout(
    barmode='overlay',
    title='Monte Carlo NPV Eloszlások Összehasonlítása (Konzervatív vs. Városmegújítás)',
    xaxis_title='NPV (M Ft)', yaxis_title='Gyakoriság',
    template=PLOTLY_TEMPLATE, height=450
)
fig1.show()

# CDF görbék összevetése
s_base = np.sort(npv_base) / 1e6
s_ren = np.sort(npv_renewal) / 1e6
p_vals = np.linspace(0, 1, len(s_base))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=s_base, y=p_vals, mode='lines', line=dict(color='#ef4444', width=3), name='Konzervatív CDF'))
fig2.add_trace(go.Scatter(x=s_ren, y=p_vals, mode='lines', line=dict(color='#10b981', width=3), name='Városmegújítás CDF'))
fig2.add_hline(y=0.05, line_color='red', line_dash='dash', annotation_text='5% (VaR szint)')
fig2.add_vline(x=0, line_color='black', line_dash='dot', annotation_text='NPV = 0')

fig2.update_layout(
    title='Kumulatív Eloszlásfüggvények (CDF) a Két Szcenárióra',
    xaxis_title='NPV (M Ft)', yaxis_title='Kumulatív Valószínűség P(X ≤ x)',
    template=PLOTLY_TEMPLATE, height=420
)
fig2.show()

### 3. Tornado Érzékenységvizsgálati Diagram

**📌 Melyik tényező a legkockázatosabb?**
A Tornado diagram megmutatja, mely változók okozzák a legnagyobb kilengést:
1. A vételár és a piac általános árszínvonala,
2. A kihasználtság / üresedési idő (Occupancy),
3. A bérleti díj ingadozása.

In [ ]:
# Tornado: az inputok ±1σ eltéréseinek hatása az NPV-re — a SZIMULÁCIÓBÓL számolva
def _npv_with(price_vol=std_price, rent_vol=std_rent, occ=0.92, cost=op_cost_ratio):
    _r = np.random.default_rng(7)
    _cov = [[price_vol**2, corr * price_vol * rent_vol],
            [corr * price_vol * rent_vol, rent_vol**2]]
    _sh = _r.multivariate_normal(mean_vec, _cov, size=2000)
    _p, _rn = _sh[:, 0], _sh[:, 1]
    _oc = np.clip(_r.normal(loc=occ, scale=0.06, size=2000), 0.70, 1.00)
    _cf = _rn * 12 * _oc * (1.0 - cost)
    _term = _p * 1.20 / ((1.0 + disc_rate) ** 20)
    return (_cf * annuity_factor + _term) - _p

base_npv_ref = np.mean(_npv_with())
factors = [
    ('Vételár volatilitás', std_price * 0.8, std_price * 1.2),
    ('Bérleti volatilitás', std_rent * 0.8, std_rent * 1.2),
    ('Kihasználtság', 0.92 - 0.06, 0.92 + 0.06),
    ('Üzemeltetési költséghányad', op_cost_ratio - 0.05, op_cost_ratio + 0.05),
]
tornado_factors = []
low_impact, high_impact = [], []
for name, lo_val, hi_val in factors:
    if 'volatilitás' in name:
        lo = np.mean(_npv_with(price_vol=lo_val)) if 'Vételár' in name else np.mean(_npv_with(rent_vol=lo_val))
        hi = np.mean(_npv_with(price_vol=hi_val)) if 'Vételár' in name else np.mean(_npv_with(rent_vol=hi_val))
    elif 'Kihasználtság' in name:
        lo = np.mean(_npv_with(occ=lo_val)); hi = np.mean(_npv_with(occ=hi_val))
    else:
        lo = np.mean(_npv_with(cost=lo_val)); hi = np.mean(_npv_with(cost=hi_val))
    tornado_factors.append(name)
    low_impact.append((lo - base_npv_ref) / 1e6)
    high_impact.append((hi - base_npv_ref) / 1e6)

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    y=tornado_factors, x=low_impact, orientation='h', name='Negatív eltérés (-1σ)', marker_color='#ef4444'
))
fig3.add_trace(go.Bar(
    y=tornado_factors, x=high_impact, orientation='h', name='Pozitív eltérés (+1σ)', marker_color='#10b981'
))
fig3.update_layout(
    title='Tornado: bemeneti változók ±1σ hatása az NPV-re (M Ft, a szimulációból számolva)',
    barmode='relative', template=PLOTLY_TEMPLATE, height=380
)
fig3.show()

# Konvergencia görbe MC-hiba sávval
step = 100
conv_iters = np.arange(step, N_ITERS + 1, step)
running_mean_base = [np.mean(npv_base[:i]) / 1e6 for i in conv_iters]
running_se_base = [np.std(npv_base[:i]) / np.sqrt(i) / 1e6 for i in conv_iters]
running_mean_ren = [np.mean(npv_renewal[:i]) / 1e6 for i in conv_iters]

fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=conv_iters, y=running_mean_base, mode='lines', line=dict(color='#ef4444', width=2), name='Konzervatív futó átlag'))
fig4.add_trace(go.Scatter(x=np.concatenate([conv_iters, conv_iters[::-1]]),
                          y=np.concatenate([[m + s for m, s in zip(running_mean_base, running_se_base)],
                                            [m - s for m, s in zip(running_mean_base, running_se_base)][::-1]]),
                          fill='toself', fillcolor='rgba(239,68,68,0.15)', line=dict(width=0),
                          name='±1 MC-SE sáv'))
fig4.add_trace(go.Scatter(x=conv_iters, y=running_mean_ren, mode='lines', line=dict(color='#10b981', width=2), name='Városmegújítás futó átlag'))
fig4.update_layout(
    title='Monte Carlo konvergencia (futó átlag ± MC-hiba sáv)',
    xaxis_title='Iterációk Száma', yaxis_title='Becsült Átlagos NPV (M Ft)',
    template=PLOTLY_TEMPLATE, height=380
)
fig4.show()


### 4. Interaktív Monte Carlo Vezérlőpult

**📌 Futtasson szimulációt!**
Állítsa be az iterációk számát és a bizonytalansági szórásokat, és kattintson a futtatás gombra!

In [ ]:
# Kliensoldali valós idejű Monte Carlo szimulációs motor
html_mc = f'''
<div id="mc_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">🎲 Valós Idejű Monte Carlo Kockázati Szimulátor</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">10 000 sztochasztikus iteráció másodpercenként a böngészőben (Box-Muller transzformáció)</p>
    </div>
    <span style="background:#dcfce7; color:#15803d; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">⚡ 10 000 Iteráció <5ms</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap:18px; margin-bottom:20px;">
    <!-- Vezérlők 1. oszlop -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="margin-bottom:12px;">
        <label style="display:block; font-size:13px; font-weight:600; color:#475569; margin-bottom:4px;">Iterációk Száma:</label>
        <select id="mc_iters" style="width:100%; padding:6px 10px; border-radius:6px; border:1px solid #cbd5e1; font-size:13px; background:#fff;" onchange="runMonteCarlo()">
          <option value="1000">1 000 minta (Villámgyors)</option>
          <option value="5000">5 000 minta</option>
          <option value="10000" selected>10 000 minta (Standard)</option>
          <option value="25000">25 000 minta (Maximális pontosság)</option>
        </select>
      </div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Vételár szórás (±%):</span> <span id="mc_lbl_p_vol" style="color:#ef4444; font-weight:700;">12%</span>
        </div>
        <input type="range" id="mc_p_vol" min="5" max="25" value="12" step="1" style="width:100%; accent-color:#ef4444;" oninput="runMonteCarlo()">
      </div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Bérleti díj szórás (±%):</span> <span id="mc_lbl_r_vol" style="color:#2563eb; font-weight:700;">10%</span>
        </div>
        <input type="range" id="mc_r_vol" min="5" max="20" value="10" step="1" style="width:100%; accent-color:#2563eb;" oninput="runMonteCarlo()">
      </div>
    </div>

    <!-- Vezérlők 2. oszlop -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Kihasználtsági ráta:</span> <span id="mc_lbl_occ" style="color:#059669; font-weight:700;">92%</span>
        </div>
        <input type="range" id="mc_occ" min="75" max="100" value="92" step="1" style="width:100%; accent-color:#059669;" oninput="runMonteCarlo()">
      </div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Éves diszkontráta:</span> <span id="mc_lbl_disc" style="color:#7c3aed; font-weight:700;">5.0%</span>
        </div>
        <input type="range" id="mc_disc" min="30" max="80" value="50" step="5" style="width:100%; accent-color:#7c3aed;" oninput="runMonteCarlo()">
      </div>

      <button onclick="runMonteCarlo()" style="width:100%; background:#2563eb; color:#fff; border:none; padding:8px 14px; border-radius:6px; font-weight:700; font-size:13px; cursor:pointer; box-shadow:0 2px 4px rgba(37,99,235,0.2);">
        🎲 Új Sztochasztikus Minta Generálása
      </button>
    </div>
  </div>

  <!-- KPI Kártyák -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:18px;">
    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Városmegújítás Várható NPV</div>
      <div id="mc_res_ren" style="font-size:26px; font-weight:800; color:#15803d; margin:4px 0;">-- M Ft</div>
      <div style="font-size:11px; color:#16a34a;">TOD felértékelődéssel</div>
    </div>

    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Konzervatív NPV</div>
      <div id="mc_res_base" style="font-size:24px; font-weight:800; color:#1d4ed8; margin:4px 0;">-- M Ft</div>
      <div style="font-size:11px; color:#2563eb;">Csak bérleti hozamból</div>
    </div>

    <div style="background:#fef2f2; border:1px solid #fca5a5; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#991b1b; text-transform:uppercase;">VaR 95% Kockázat</div>
      <div id="mc_res_var" style="font-size:24px; font-weight:800; color:#dc2626; margin:4px 0;">-- M Ft</div>
      <div style="font-size:11px; color:#b91c1c;">5%-os legrosszabb küszöb</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">P(NPV > 0) Sikeresség</div>
      <div id="mc_res_prob" style="font-size:24px; font-weight:800; color:#7e22ce; margin:4px 0;">--%</div>
      <div style="font-size:11px; color:#9333ea;">Pozitív hozam valószínűsége</div>
    </div>
  </div>

  <!-- SVG Eloszlás Histrogram -->
  <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:8px; padding:14px; text-align:center;">
    <div style="font-size:12px; font-weight:700; color:#475569; margin-bottom:8px;">Városmegújítási NPV Szimulált Eloszlása (Valós idejű hisztogram):</div>
    <svg id="mc_svg_chart" viewBox="0 0 700 180" style="width:100%; height:180px; max-width:700px;"></svg>
    <div style="display:flex; justify-content:center; gap:20px; font-size:11px; color:#64748b; margin-top:6px;">
      <span>🟢 Zöld oszlopok: Nyereséges tartomány (NPV > 0)</span>
      <span>🔴 Piros függőleges vonal: VaR 95% küszöb</span>
    </div>
  </div>
</div>

<script>
(function() {{
  const base_p = {float(base_price)};
  const base_r = {float(base_rent)};
  const op_cost = 0.15;

  function update() {{
    const N = parseInt(document.getElementById('mc_iters').value) || 10000;
    const p_vol = parseFloat(document.getElementById('mc_p_vol').value) / 100;
    const r_vol = parseFloat(document.getElementById('mc_r_vol').value) / 100;
    const occ = parseFloat(document.getElementById('mc_occ').value) / 100;
    const disc = parseFloat(document.getElementById('mc_disc').value) / 1000;

    document.getElementById('mc_lbl_p_vol').innerText = Math.round(p_vol * 100) + '%';
    document.getElementById('mc_lbl_r_vol').innerText = Math.round(r_vol * 100) + '%';
    document.getElementById('mc_lbl_occ').innerText = Math.round(occ * 100) + '%';
    document.getElementById('mc_lbl_disc').innerText = (disc * 100).toFixed(1) + '%';

    const annuity = (1.0 - Math.pow(1.0 + disc, -20)) / disc;
    const term_base_factor = 1.20 / Math.pow(1.0 + disc, 20);
    const term_ren_factor = 1.80 / Math.pow(1.0 + disc, 20);

    const corr = 0.65;
    const corr_inv = Math.sqrt(1 - corr * corr);

    let ren_arr = new Float64Array(N);
    let sum_base = 0, sum_ren = 0, pos_ren = 0;

    for (let i = 0; i < N; i++) {{
      let u1 = Math.max(1e-7, Math.random());
      let u2 = Math.random();
      let z1 = Math.sqrt(-2.0 * Math.log(u1)) * Math.cos(2.0 * Math.PI * u2);
      let z2 = Math.sqrt(-2.0 * Math.log(u1)) * Math.sin(2.0 * Math.PI * u2);

      let p_s = base_p * (1.0 + p_vol * z1);
      let r_s = base_r * (1.0 + r_vol * (corr * z1 + corr_inv * z2));
      let cf = r_s * 12 * occ * (1.0 - op_cost);

      let nb = (cf * annuity + p_s * term_base_factor) - p_s;
      let nr = (cf * annuity + p_s * term_ren_factor) - p_s;

      ren_arr[i] = nr;
      sum_base += nb;
      sum_ren += nr;
      if (nr > 0) pos_ren++;
    }}

    ren_arr.sort();
    const mean_ren = (sum_ren / N) / 1e6;
    const mean_base = (sum_base / N) / 1e6;
    const var95 = ren_arr[Math.floor(N * 0.05)] / 1e6;
    const prob_ren = (pos_ren / N) * 100;

    document.getElementById('mc_res_ren').innerText = (mean_ren > 0 ? '+' : '') + mean_ren.toFixed(1) + ' M Ft';
    document.getElementById('mc_res_base').innerText = (mean_base > 0 ? '+' : '') + mean_base.toFixed(1) + ' M Ft';
    document.getElementById('mc_res_var').innerText = (var95 > 0 ? '+' : '') + var95.toFixed(1) + ' M Ft';
    document.getElementById('mc_res_prob').innerText = prob_ren.toFixed(1) + '%';

    // SVG hisztogram kirajzolása
    const min_v = ren_arr[Math.floor(N * 0.01)] / 1e6;
    const max_v = ren_arr[Math.floor(N * 0.99)] / 1e6;
    const BINS = 35;
    const bin_w = (max_v - min_v) / BINS;
    let counts = new Int32Array(BINS);
    for (let i = 0; i < N; i++) {{
      let v = ren_arr[i] / 1e6;
      if (v >= min_v && v < max_v) {{
        let b = Math.floor((v - min_v) / bin_w);
        if (b >= 0 && b < BINS) counts[b]++;
      }}
    }}
    let max_c = 1;
    for (let b = 0; b < BINS; b++) if (counts[b] > max_c) max_c = counts[b];

    const svg = document.getElementById('mc_svg_chart');
    if (!svg) return;
    let svg_inner = '';
    const W = 700, H = 160, PAD = 30;
    const chart_w = W - 2 * PAD;
    const chart_h = H - PAD;
    const bar_pixel_w = chart_w / BINS;

    for (let b = 0; b < BINS; b++) {{
      let val = min_v + (b + 0.5) * bin_w;
      let bh = (counts[b] / max_c) * (chart_h - 10);
      let x = PAD + b * bar_pixel_w;
      let y = chart_h - bh;
      let color = val >= 0 ? '#10b981' : '#ef4444';
      svg_inner += `<rect x="${{x}}" y="${{y}}" width="${{bar_pixel_w - 2}}" height="${{bh}}" fill="${{color}}" opacity="0.75"><title>${{val.toFixed(1)}} M Ft: ${{counts[b]}} db</title></rect>`;
    }}

    // Zéró vonal (NPV = 0)
    if (min_v < 0 && max_v > 0) {{
      let zx = PAD + ((0 - min_v) / (max_v - min_v)) * chart_w;
      svg_inner += `<line x1="${{zx}}" y1="10" x2="${{zx}}" y2="${{chart_h}}" stroke="#000" stroke-width="2" stroke-dasharray="4"/>`;
      svg_inner += `<text x="${{zx + 4}}" y="20" font-size="10" font-weight="700" fill="#000">NPV = 0</text>`;
    }}

    // VaR 95 vonal
    let vx = PAD + ((var95 - min_v) / (max_v - min_v)) * chart_w;
    if (vx >= PAD && vx <= W - PAD) {{
      svg_inner += `<line x1="${{vx}}" y1="10" x2="${{vx}}" y2="${{chart_h}}" stroke="#dc2626" stroke-width="2"/>`;
      svg_inner += `<text x="${{vx - 60}}" y="35" font-size="10" font-weight="700" fill="#dc2626">VaR: ${{var95.toFixed(1)}}M</text>`;
    }}

    // Tengelyvonal
    svg_inner += `<line x1="${{PAD}}" y1="${{chart_h}}" x2="${{W - PAD}}" y2="${{chart_h}}" stroke="#94a3b8" stroke-width="1"/>`;
    svg_inner += `<text x="${{PAD}}" y="${{chart_h + 16}}" font-size="10" fill="#64748b">${{min_v.toFixed(0)}} M Ft</text>`;
    svg_inner += `<text x="${{W - PAD - 40}}" y="${{chart_h + 16}}" font-size="10" fill="#64748b">${{max_v.toFixed(0)}} M Ft</text>`;

    svg.innerHTML = svg_inner;
  }}

  window.runMonteCarlo = update;
  setTimeout(update, 50);
}})();
</script>
'''
display(HTML(html_mc))